# Run multimodal candidate profile analysis

This notebook runs `build_multimodal_candidate_profiles.py` using the CSV files inside your `data/` folder and writes all results to a new folder inside `outputs/`.

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd

PROJECT_ROOT = Path('.').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'multimodal_profile_analysis'
SCRIPT_PATH = PROJECT_ROOT / 'build_multimodal_candidate_profiles.py'

print('Project root:', PROJECT_ROOT)
print('Data folder:', DATA_DIR)
print('Output folder:', OUTPUT_DIR)
print('Script:', SCRIPT_PATH)

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f'Could not find {SCRIPT_PATH}. Put build_multimodal_candidate_profiles.py in the project root.')
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Could not find data folder: {DATA_DIR}')

## Find your input CSVs

This cell accepts filenames with or without `(1)` suffixes, as long as they are in `data/`.

In [ ]:
def find_one(patterns, label):
    matches = []
    for pattern in patterns:
        matches.extend(DATA_DIR.glob(pattern))
    matches = sorted(set(matches))
    if not matches:
        print(f'Files currently in data/:')
        for p in sorted(DATA_DIR.glob('*.csv')):
            print(' -', p.name)
        raise FileNotFoundError(f'Could not find {label}. Tried patterns: {patterns}')
    print(f'{label}:', matches[0])
    return matches[0]

MAPPING_CSV = find_one(['01_candidate_name_mapping_all_debates*.csv'], 'candidate mapping CSV')
MOVEMENT_CSV = find_one(['02_movement_10s_all_debates*.csv'], 'movement CSV')
EMOTIONS_CSV = find_one(['03_emotions_10s_all_debates*.csv'], 'emotions CSV')
TOPICS_CSV = find_one(['04_topics_10s_all_debates*.csv'], 'topics CSV')
AUDIO_CSV = find_one(['audio_events_by_second_all_debates_clean*.csv'], 'audio events CSV')

## Run the analysis

This creates the merged 10-second window file, candidate profiles, topic profiles, correlations, peak events, and the text assessment file.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--input-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--movement', str(MOVEMENT_CSV),
    '--emotions', str(EMOTIONS_CSV),
    '--topics', str(TOPICS_CSV),
    '--audio', str(AUDIO_CSV),
]

print('Running command:')
print(' '.join(cmd))

result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Analysis script failed.')

print('Done. Outputs saved to:', OUTPUT_DIR)

## List output files

In [ ]:
for p in sorted(OUTPUT_DIR.glob('*')):
    print(p.name)

## Preview main merged table

In [ ]:
merged_path = OUTPUT_DIR / '01_multimodal_10s_windows.csv'
merged = pd.read_csv(merged_path)
print('Shape:', merged.shape)
display(merged.head(20))
print('Columns:')
print(list(merged.columns))

## Preview global candidate profiles

In [ ]:
profiles_path = OUTPUT_DIR / '02_candidate_profiles_global.csv'
profiles = pd.read_csv(profiles_path)
print('Shape:', profiles.shape)

preview_cols = [
    'candidate', 'n_debates', 'total_visible_seconds', 'total_speaking_seconds',
    'avg_movement', 'avg_hand_movement', 'avg_non_neutral_emotion_score',
    'avg_speechrate', 'avg_speechrate_z', 'avg_pitch_variability', 'avg_pitchvar_z',
    'interruptions_per_100_speaking_sec', 'top_emotions', 'top_topics',
    'expressiveness_score', 'composure_score'
]
preview_cols = [c for c in preview_cols if c in profiles.columns]
display(profiles[preview_cols].head(50))

## Preview candidate-topic profiles

In [ ]:
topic_profiles_path = OUTPUT_DIR / '04_candidate_topic_profiles.csv'
topic_profiles = pd.read_csv(topic_profiles_path)
print('Shape:', topic_profiles.shape)
display(topic_profiles.head(50))

## Preview strongest correlations

In [ ]:
global_corr_path = OUTPUT_DIR / '05_global_correlations.csv'
global_corr = pd.read_csv(global_corr_path)
print('Shape:', global_corr.shape)
if not global_corr.empty:
    global_corr['abs_spearman'] = global_corr['spearman'].abs()
    display(global_corr.sort_values('abs_spearman', ascending=False).head(30))
else:
    print('No global correlations were produced. Usually this means too few non-missing overlapping rows.')

## Preview multimodal peak events

In [ ]:
peak_path = OUTPUT_DIR / '08_peak_events.csv'
peak_events = pd.read_csv(peak_path)
print('Shape:', peak_events.shape)
display(peak_events.head(50))

## Read the automatic assessment text

In [ ]:
assessment_path = OUTPUT_DIR / '09_analysis_assessments.txt'
print(assessment_path.read_text(encoding='utf-8')[:6000])